# Explorer les solveurs de transport et de diffusion

Ce notebook relie les **équations physiques**, les **algorithmes effectivement implémentés** et leurs **résultats numériques**. Tous les exemples utilisent un même réacteur cœur–réflecteur sous-critique afin de comparer les solutions stationnaires et le démarrage depuis un flux nul.

Exécuter les cellules dans l'ordre avec le noyau Python du projet. Depuis la racine, installer l'environnement avec `uv sync --group notebooks`. Le premier appel aux noyaux Numba peut inclure leur compilation ; les temps de ce notebook ne constituent pas un benchmark.

| Étape | Interface | Question posée |
| --- | --- | --- |
| 1 | `solve_stationnary_source` | Quel flux produit une source imposée, sans production de fission ? |
| 2 | `StatDiffusionSolver.solve` | Quel équilibre obtient-on avec la multiplication par fission ? |
| 3 | `criticity_factor` | Quel est le facteur de multiplication du problème propre de diffusion ? |
| 4 | `EulerDiffusionFissionSolver.solve` | Comment le flux de diffusion rejoint-il cet équilibre ? |
| 5 | `StatTransportProblem.solve` | Quel flux obtient-on en conservant les directions des neutrons ? |
| 6 | `StatTransportProblem.solve_flux_only` | Comment résoudre le même problème avec l'itération entièrement compilée ? |
| 7 | `EulerTransportProblem.solve` | Comment évolue le flux angulaire dans le temps ? |
| 8 | Assemblage pédagogique des outils SN | Comment construire une itération de criticité en transport ? |

La dernière étape est une boucle écrite dans ce notebook : le module SN expose la résolution à source fixée et le calcul du résidu propre, mais pas de solveur de criticité complet.

In [ ]:
from pathlib import Path
import sys

# Fonctionne depuis la racine du dépôt ou depuis notebooks/.
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (cwd, *cwd.parents) if (path / "transport").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Ouvrir ce notebook depuis le dépôt transport.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from transport import Material1D, Reactor1D, UniformMesh1D, SNQuadrature
from transport.diffusion import (
    StatDiffusionSolver,
    EulerDiffusionFissionSolver,
    assemble_diffusion_loss_matrix,
    assemble_fission_matrix,
    diffusion_current,
    criticity_factor,
    solve_stationnary_source,
)
from transport.sn import StatTransportProblem, EulerTransportProblem

%matplotlib inline
plt.rcParams.update({
    "figure.figsize": (10, 4),
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.18,
    "font.size": 11,
})
np.set_printoptions(precision=5, suppress=True)

## 0. Géométrie, matériaux et conventions

On travaille sur $x\in[-L,L]$ avec un cœur fissile $|x|<a$ et deux réflecteurs. Les neutrons ont une vitesse unique $v$ ; les sources, la diffusion des directions et la production de fission sont isotropes.

$$
\Sigma_t=\Sigma_s+\Sigma_c+\Sigma_f,\qquad
\Sigma_a=\Sigma_c+\Sigma_f,\qquad
\Sigma_p=\Sigma_s+\nu\Sigma_f,\qquad
D=\frac{1}{3\Sigma_t}.
$$

| Symbole | Sens | Représentation |
| --- | --- | --- |
| $\psi(x,\mu,t)$ | Flux angulaire, $\mu\in[-1,1]$ | `psi[m, i]` |
| $\phi(x,t)=\int_{-1}^1\psi\,d\mu$ | Flux scalaire | `phi[i]` |
| $J=\int_{-1}^1\mu\psi\,d\mu$ | Courant en transport | Aux centres des cellules |
| $J=-D\,\partial_x\phi$ | Courant en diffusion | Aux faces des cellules |
| $q(x)$ | Source externe intégrée sur les directions | `mesh.source` |
| $h=2L/N$ | Largeur des cellules | `mesh.h` |

**Les trois représentations de la source.** Pour une même source physique $q$, le second membre des volumes finis de diffusion est $hq$, tandis que la source angulaire isotrope du transport est $q/2$ puisque $\int_{-1}^{1}(q/2)\,d\mu=q$. Les exemples passent ces quantités explicitement.

Les valeurs ci-dessous sont un cas pédagogique en unités cohérentes arbitraires ; elles ne décrivent pas une installation réelle. Le choix $\Sigma_a>\nu\Sigma_f$ dans chaque région permet un équilibre positif avec une source externe.

In [ ]:
core = Material1D(
    sigma_s=0.70, sigma_c=0.06, sigma_f=0.02, nu=2.4, source=0.03
)
reflector = Material1D(
    sigma_s=0.80, sigma_c=0.04, sigma_f=0.0, nu=0.0, source=0.0
)
reactor = Reactor1D(
    half_width=10.0,
    core_half_width=6.0,
    speed=1.0,
    core=core,
    reflector=reflector,
)
N_CELLS = 120
N_DIRECTIONS = 8
BOUNDARY = "extrapolated"
PHI_TOL = 1e-10

mesh = UniformMesh1D.from_reactor(reactor, n_cells=N_CELLS)
quadrature = SNQuadrature.gauss_legendre(N_DIRECTIONS)
x = mesh.centers
q = mesh.source
q_angular = 0.5 * q
fission = mesh.nu * mesh.Sigma_f

assert np.all(mesh.Sigma_a > fission)
assert np.isclose(quadrature.weights.sum(), 2.0)
assert np.any(np.isclose(mesh.faces, reactor.core_half_width))
assert np.any(np.isclose(mesh.faces, -reactor.core_half_width))

print(f"Maillage : {mesh.n_cells} cellules, h = {mesh.h:.4f}")
print(f"Quadrature : S{quadrature.order}, somme des poids = {quadrature.weights.sum():.1f}")
print(f"Flux scalaire : ({mesh.n_cells},) ; angulaire : ({quadrature.order}, {mesh.n_cells})")

In [ ]:
def shade_reactor(ax):
    """Repère le cœur et les réflecteurs sur un axe spatial."""
    L, a = reactor.half_width, reactor.core_half_width
    ax.axvspan(-L, -a, color="#4f8ab8", alpha=0.10, zorder=0)
    ax.axvspan(-a, a, color="#e7a351", alpha=0.13, zorder=0)
    ax.axvspan(a, L, color="#4f8ab8", alpha=0.10, zorder=0)
    for interface in (-a, a):
        ax.axvline(interface, color="#64748b", lw=0.8, ls=":")
    ax.set_xlim(-L, L)
    ax.set_xlabel("Position x")


def relative_gap(values, reference):
    """Mesure l'écart relatif en norme infinie."""
    return np.linalg.norm(values - reference, ord=np.inf) / max(
        np.linalg.norm(reference, ord=np.inf), np.finfo(float).tiny
    )


fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), layout="constrained")
shade_reactor(axes[0])
axes[0].step(x, mesh.Sigma_a, where="mid", label=r"Absorption $\Sigma_a$")
axes[0].step(x, fission, where="mid", label=r"Production $\nu\Sigma_f$")
axes[0].set_ylabel("Coefficient")
axes[0].set_title("Réflecteur | Cœur | Réflecteur")
axes[0].legend()
shade_reactor(axes[1])
axes[1].step(x, q, where="mid", color="#d97706")
axes[1].set_ylabel("Source scalaire q")
axes[1].set_title("Source externe localisée dans le cœur")
plt.show()

### L'opérateur de diffusion commun aux quatre premiers exemples

Dans la cellule $i$, l'intégration de $-\partial_x(D\,\partial_x\phi)+\Sigma_a\phi$ donne

$$
(A\phi)_i=J_{i+1/2}-J_{i-1/2}+h\Sigma_{a,i}\phi_i,
\qquad
J_{i+1/2}=-D_{i+1/2}\frac{\phi_{i+1}-\phi_i}{h}.
$$

À une face intérieure, le code utilise la moyenne harmonique

$$
D_{i+1/2}=\frac{2D_iD_{i+1}}{D_i+D_{i+1}}.
$$

Les conditions extrapolées choisies ici utilisent $\delta_L=2.1312D_0$ et $\delta_R=2.1312D_{N-1}$ :

$$
J_{-1/2}=-\frac{D_0\phi_0}{h/2+\delta_L},
\qquad
J_{N-1/2}=\frac{D_{N-1}\phi_{N-1}}{h/2+\delta_R}.
$$

La matrice de production est $F=\operatorname{diag}(h\nu\Sigma_f)$ et le second membre externe est $b=hq$. Les matrices sont creuses ; les solveurs les factorisent avec une décomposition LU.

Cette fermeture de diffusion approxime une frontière entourée de vide. En transport, le vide sera imposé directement par l'absence de flux **entrant**.

In [ ]:
A = assemble_diffusion_loss_matrix(mesh, boundary=BOUNDARY)
F = assemble_fission_matrix(mesh)
b = mesh.h * q

print(f"A : {A.shape}, {A.nnz} coefficients non nuls")
print(f"F : {F.shape} ; b : {b.shape}")

## 1. Source imposée sans production de fission — `solve_stationnary_source`

**Problème.** Déterminer le flux entretenu par $q$, avec diffusion et absorption :

$$
-\partial_x(D\,\partial_x\phi)+\Sigma_a\phi=q
\quad\Longrightarrow\quad
A\phi=b.
$$

La contribution de $\Sigma_f$ à l'absorption reste dans $\Sigma_a$ ; c'est la **production** $\nu\Sigma_f\phi$ qui est absente de ce problème.

**Algorithme.** Assembler $A$, factoriser $A=LU$, puis résoudre successivement les systèmes triangulaires pour obtenir $\phi=A^{-1}b$. L'inverse n'est pas construit.

**API.** La fonction attend déjà $b=hq$, renvoie uniquement `phi`, et utilise les conditions extrapolées par défaut. Elle refactorise $A$ à chaque appel. Voici un usage explicite de cette fonction utilitaire.

In [ ]:
phi_source = solve_stationnary_source(mesh, b)
source_residual = np.linalg.norm(A @ phi_source - b, ord=np.inf)
assert np.all(np.isfinite(phi_source)) and np.all(phi_source >= 0.0)
assert source_residual < 1e-10

print(f"Résidu ||A phi - b||∞ = {source_residual:.3e}")
fig, ax = plt.subplots(layout="constrained")
shade_reactor(ax)
ax.plot(x, phi_source, color="#2563eb", label="Diffusion sans production de fission")
ax.set_ylabel("Flux scalaire")
ax.set_title("Équilibre entre la source, l'absorption et les fuites")
ax.legend()
plt.show()

## 2. Diffusion stationnaire avec fission - `StatDiffusionSolver`

**Problème.** La source externe et les neutrons produits par fission entretiennent maintenant le flux :

$$
-\partial_x(D\,\partial_x\phi)+\Sigma_a\phi
=\nu\Sigma_f\phi+q
\quad\Longrightarrow\quad
(A-F)\phi=b.
$$

**Algorithme.** Factoriser $A-F$, résoudre le système linéaire, puis reconstruire le courant aux faces et le résidu de bilan **par unité de longueur** :

$$
r_i=\frac{J_{i+1/2}-J_{i-1/2}}{h}
+\Sigma_{a,i}\phi_i-\nu_i\Sigma_{f,i}\phi_i-q_i.
$$

Le solveur contrôle les dimensions, la finitude, la non-négativité à la tolérance près et $\|r\|_\infty$. Le maillage est fourni à `solve(mesh)` ; le constructeur règle les conditions au bord.

**À observer.** Pour ce cas sous-critique, la production de fission augmente le flux par rapport à l'exemple précédent, tout en permettant un équilibre fini.

In [ ]:
diffusion_solver = StatDiffusionSolver(boundary=BOUNDARY)
diffusion_stationary = diffusion_solver.solve(mesh)
phi_diffusion = diffusion_stationary.phi

np.testing.assert_allclose((A - F) @ phi_diffusion, b, rtol=1e-9, atol=1e-11)
assert np.all(phi_diffusion >= phi_source - 1e-12)

# Bilan intégré : fuites nettes + absorptions - production - source = 0.
global_balance = (
    diffusion_stationary.current[-1] - diffusion_stationary.current[0]
    + mesh.h * np.sum((mesh.Sigma_a - fission) * phi_diffusion - q)
)
assert abs(global_balance) < 1e-10
print(f"Résidu local ||r||∞ = {diffusion_stationary.residual_norm_inf:.3e}")
print(f"Bilan global       = {global_balance:.3e}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
shade_reactor(axes[0])
axes[0].plot(x, phi_source, "--", label="Sans production de fission")
axes[0].plot(x, phi_diffusion, color="#d97706", label="Avec production de fission")
axes[0].set_ylabel("Flux scalaire")
axes[0].set_title("Effet de la multiplication neutronique")
axes[0].legend()
shade_reactor(axes[1])
axes[1].plot(mesh.faces, diffusion_stationary.current, color="#0d9488")
axes[1].axhline(0, color="#64748b", lw=0.8)
axes[1].set_ylabel("Courant J aux faces")
axes[1].set_title("Courants sortants vers les deux bords")
plt.show()

## 3. Criticité en diffusion — `criticity_factor`

**Problème.** On retire la source externe et on cherche un facteur $k$ et un flux non nul tels que

$$
-\partial_x(D\,\partial_x\phi)+\Sigma_a\phi
=\frac{\nu\Sigma_f}{k}\phi
\quad\Longleftrightarrow\quad
A\phi=\frac{1}{k}F\phi
\quad\Longleftrightarrow\quad
A^{-1}F\phi=k\phi.
$$

Le mode dominant fournit $k_{\mathrm{eff}}$ : inférieur à 1, le système est sous-critique ; égal à 1, critique ; supérieur à 1, surcritique dans ce modèle. La présence de `mesh.source` n'affecte pas ce calcul : cette fonction ne l'utilise pas.

**Algorithme réellement utilisé.** La production est $P(\phi)=\sum_i h\nu_i\Sigma_{f,i}\phi_i$. Après normalisation initiale $P(\phi^{(0)})=1$, l'itération de puissance s'écrit

$$
A\widetilde\phi^{(n+1)}=F\phi^{(n)},\qquad
k^{(n+1)}=P(\widetilde\phi^{(n+1)}),\qquad
\phi^{(n+1)}=\frac{\widetilde\phi^{(n+1)}}{k^{(n+1)}}.
$$

Une seule factorisation de $A$ est réutilisée. L'arrêt impose simultanément

$$
\frac{|k^{(n+1)}-k^{(n)}|}{|k^{(n+1)}|}<\texttt{ktol},
\qquad
\frac{\|A\phi-F\phi/k\|_\infty}
{\max(\|A\phi\|_\infty,\|F\phi/k\|_\infty,\varepsilon_{\rm tiny})}
<\texttt{residual\_tol}.
$$

Le flux propre est normalisé par sa production : son amplitude n'est pas celle du flux stationnaire entretenu par $q$.

In [ ]:
criticality = criticity_factor(
    mesh, ktol=1e-10, residual_tol=1e-9, boundary=BOUNDARY
)
k_diffusion = criticality.k_eff
eigen_lhs = A @ criticality.phi
eigen_rhs = (F @ criticality.phi) / k_diffusion
eigen_residual = np.linalg.norm(eigen_lhs - eigen_rhs, ord=np.inf) / max(
    np.linalg.norm(eigen_lhs, ord=np.inf),
    np.linalg.norm(eigen_rhs, ord=np.inf),
    np.finfo(float).tiny,
)

assert 0 < k_diffusion < 1
assert eigen_residual < 1e-9
np.testing.assert_allclose(np.sum(F @ criticality.phi), 1.0, rtol=1e-12)
print(f"k_eff = {k_diffusion:.8f} : cas sous-critique")
print(f"Itérations = {criticality.iterations} ; résidu relatif = {eigen_residual:.3e}")

fig, ax = plt.subplots(layout="constrained")
shade_reactor(ax)
ax.plot(x, criticality.phi / criticality.phi.max(), label="Mode propre de diffusion")
ax.plot(x, phi_diffusion / phi_diffusion.max(), "--", label="Flux avec source externe")
ax.set_ylabel("Flux normalisé par son maximum")
ax.set_title("Deux problèmes distincts : criticité et équilibre avec source")
ax.legend()
plt.show()

## 4. Diffusion transitoire — `EulerDiffusionFissionSolver`

**Problème.** À partir de $\phi(x,0)=0$, on allume la source constante $q$ :

$$
\frac{1}{v}\partial_t\phi-\partial_x(D\,\partial_x\phi)
+\Sigma_a\phi=\nu\Sigma_f\phi+q.
$$

**Euler implicite.** Avec $M=\frac{h}{v\Delta t}I$,

$$
M(\phi^{n+1}-\phi^n)+(A-F)\phi^{n+1}=b,
\qquad
(M+A-F)\phi^{n+1}=M\phi^n+b.
$$

Le système est factorisé dans le constructeur et sa factorisation est réutilisée à chaque pas. On choisit $T/\Delta t$ entier ; les tableaux `phi` et `current` incluent l'état initial.

**À observer.** Le flux monte progressivement vers la solution de `StatDiffusionSolver`. Le rapprochement vers l'équilibre ne mesure pas l'ordre de précision en temps : celui-ci nécessiterait une étude en $\Delta t$.

**Détail de la sortie actuelle.** Le solveur remplit le courant aux faces intérieures et laisse les deux faces de bord à zéro dans `result.current`. Pour étudier les fuites, on reconstruit ici le courant complet avec `diffusion_current` et la même condition extrapolée.

In [ ]:
ENDTIME = 300.0
TIMESTEP = 2.0
N_STEPS = int(round(ENDTIME / TIMESTEP))
times = np.arange(N_STEPS + 1) * TIMESTEP
snapshot_steps = np.unique(np.rint(
    np.array([0.0, 1 / 30, 0.1, 1 / 3, 1.0]) * N_STEPS
).astype(int))
phi_initial = np.zeros(mesh.n_cells)

diffusion_time_solver = EulerDiffusionFissionSolver(
    mesh=mesh,
    velocity=reactor.speed,
    endtime=ENDTIME,
    timestep=TIMESTEP,
    phi0=phi_initial,
    boundary=BOUNDARY,
)
diffusion_time = diffusion_time_solver.solve()

mass = mesh.h / (reactor.speed * TIMESTEP)
diffusion_step_residuals = (
    mass * np.diff(diffusion_time.phi, axis=0)
    + ((A - F) @ diffusion_time.phi[1:].T).T
    - b[None, :]
)
diffusion_time_gap = np.max(
    np.abs(diffusion_time.phi - phi_diffusion[None, :]), axis=1
) / np.max(np.abs(phi_diffusion))
diffusion_final_current = diffusion_current(
    mesh, diffusion_time.phi[-1], boundary=BOUNDARY
)

assert diffusion_time.phi.shape == (N_STEPS + 1, mesh.n_cells)
assert diffusion_time.is_finite and diffusion_time.is_positive
assert np.max(np.abs(diffusion_step_residuals)) < 1e-10
assert diffusion_time_gap[-1] < diffusion_time_gap[0]
np.testing.assert_allclose(
    diffusion_time.current[-1, 1:-1], diffusion_final_current[1:-1]
)
print("Historique phi :", diffusion_time.phi.shape)
print(f"Résidu maximal d'un pas Euler : {np.max(np.abs(diffusion_step_residuals)):.3e}")
print(f"Écart relatif final au stationnaire : {diffusion_time_gap[-1]:.3e}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), layout="constrained")
shade_reactor(axes[0])
for step in snapshot_steps:
    axes[0].plot(x, diffusion_time.phi[step], label=f"t = {times[step]:g}")
axes[0].plot(x, phi_diffusion, "k--", lw=1.3, label="Stationnaire")
axes[0].set_ylabel("Flux scalaire")
axes[0].set_title("Allumage de la source")
axes[0].legend(fontsize=8)

image = axes[1].pcolormesh(x, times, diffusion_time.phi, shading="auto", cmap="magma")
axes[1].set_xlabel("Position x")
axes[1].set_ylabel("Temps")
axes[1].set_title("Évolution spatio-temporelle")
fig.colorbar(image, ax=axes[1], label="Flux scalaire")

axes[2].semilogy(times, diffusion_time_gap, color="#2563eb")
axes[2].set_xlabel("Temps")
axes[2].set_ylabel("Écart relatif au stationnaire")
axes[2].set_title("Approche de l'équilibre")
plt.show()

## 5. Transport stationnaire SN — `StatTransportProblem.solve`

**Problème.** On conserve la direction $\mu$ :

$$
\mu\,\partial_x\psi+\Sigma_t\psi
=\frac{\Sigma_p}{2}\phi+\frac{q}{2},
\qquad
\phi=\int_{-1}^{1}\psi\,d\mu.
$$

Le vide correspond à $\psi(-L,\mu>0)=0$ et $\psi(L,\mu<0)=0$. Le flux sortant reste libre.

**Quadrature angulaire.** Avec les nœuds et poids de Gauss–Legendre,

$$
\phi_i\simeq\sum_{m=1}^{M}w_m\psi_{m,i},
\qquad
J_i\simeq\sum_{m=1}^{M}w_m\mu_m\psi_{m,i}.
$$

Ici $M=8$ directions pour $N=120$ cellules ; la notation SN désigne la famille des méthodes aux ordonnées discrètes.

**Itération sur la source et balayage amont.** À l'itération $\ell$,

$$
s_i^{(\ell)}=\frac{\Sigma_{p,i}}2\phi_i^{(\ell)}+\frac{q_i}{2},
\qquad
\psi_{m,i}^{(\ell+1)}
=\frac{|\mu_m|\psi_{m,\mathrm{amont}(i)}^{(\ell+1)}
+h\,s_i^{(\ell)}}{|\mu_m|+h\Sigma_{t,i}}.
$$

Le balayage parcourt les cellules de gauche à droite pour $\mu_m>0$, et de droite à gauche pour $\mu_m<0$. Une fois les deux demi-espaces traités, on recalcule $\phi$ par quadrature. Le critère d'arrêt est

$$
e_\ell=
\frac{\|\phi^{(\ell+1)}-\phi^{(\ell)}\|_\infty}
{\max(\|\phi^{(\ell)}\|_\infty,\varepsilon_{\rm tiny})}
<\texttt{phi\_tol}.
$$

Ce critère mesure la variation des itérés. On calcule aussi ci-dessous un résidu indépendant de l'équation discrète. `fixed_source` est une source **angulaire** $q/2$ ; `source_operator` est ici $\Sigma_p$.

**Comparaison avec la diffusion.** Les deux modèles utilisent la même source scalaire et les mêmes matériaux. Leurs fermetures angulaires, conditions de vide et erreurs de discrétisation diffèrent : leurs flux ne doivent pas être supposés identiques.

In [ ]:
sn_problem = StatTransportProblem(mesh=mesh, quadrature=quadrature)
psi_guess = np.full((quadrature.order, mesh.n_cells), 0.5)

sn_stationary = sn_problem.solve(
    psi0=psi_guess,
    fixed_source=q_angular,
    source_operator=mesh.Sigma_p,
    phi_tol=PHI_TOL,
)
phi_sn = sn_stationary.phi

# Reconstruction indépendante de l'amont, avec flux entrant nul.
psi_upstream = np.zeros_like(sn_stationary.psi)
half = quadrature.order // 2
psi_upstream[:half, :-1] = sn_stationary.psi[:half, 1:]  # mu < 0
psi_upstream[half:, 1:] = sn_stationary.psi[half:, :-1]   # mu > 0
sn_balance_residual = (
    np.abs(quadrature.nodes)[:, None]
    * (sn_stationary.psi - psi_upstream) / mesh.h
    + mesh.Sigma_t[None, :] * sn_stationary.psi
    - 0.5 * mesh.Sigma_p[None, :] * phi_sn[None, :]
    - q_angular[None, :]
)
sn_residual_max = np.max(np.abs(sn_balance_residual))
diffusion_sn_gap = relative_gap(phi_diffusion, phi_sn)

assert sn_stationary.is_finite and sn_stationary.is_positive
assert sn_residual_max < 1e-8
np.testing.assert_allclose(quadrature.weights @ sn_stationary.psi, phi_sn)
print(f"Itérations sur la source : {sn_stationary.iterations}")
print(f"Dernière variation relative du flux : {sn_stationary.errors[-1]:.3e}")
print(f"Résidu maximal de l'équation SN : {sn_residual_max:.3e}")
print(f"Écart diffusion / SN : {diffusion_sn_gap:.2%}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), layout="constrained")
shade_reactor(axes[0])
axes[0].plot(x, phi_diffusion, "--", color="#d97706", label="Diffusion")
axes[0].plot(x, phi_sn, color="#2563eb", label=f"Transport S{quadrature.order}")
axes[0].set_ylabel("Flux scalaire")
axes[0].set_title("Même source physique")
axes[0].legend()

image = axes[1].pcolormesh(
    x, quadrature.nodes, sn_stationary.psi, shading="auto", cmap="viridis"
)
axes[1].set_xlabel("Position x")
axes[1].set_ylabel(r"Direction $\mu$")
axes[1].set_title("Flux angulaire")
fig.colorbar(image, ax=axes[1], label=r"$\psi$")

axes[2].semilogy(np.arange(1, sn_stationary.iterations + 1), sn_stationary.errors)
axes[2].axhline(PHI_TOL, color="#d97706", ls="--", label="Tolérance")
axes[2].set_xlabel("Itération sur la source")
axes[2].set_ylabel("Variation relative de phi")
axes[2].set_title("Convergence du point fixe")
axes[2].legend()
plt.show()

fig, ax = plt.subplots(layout="constrained")
shade_reactor(ax)
ax.plot(mesh.faces, diffusion_stationary.current, "--", label="Diffusion : faces")
ax.plot(x, sn_stationary.current, label="Transport : centres")
ax.set_ylabel("Courant")
ax.set_title("Deux localisations discrètes du courant")
ax.legend()
plt.show()

## 6. Même problème, itération compilée — `solve_flux_only`

Les équations, balayages et critère d'arrêt sont ceux de la section précédente. Cette méthode déplace **toute l'itération sur la source** dans `solve_flux_only_numba` et réutilise ses tableaux de travail.

Elle renvoie le tuple `(psi, phi, iterations, error)`, sans calcul du courant ni historique complet des erreurs. On utilise la même initialisation et les mêmes sources pour comparer les deux chemins.

L'option `assume_valid=True` supprime les conversions et vérifications d'entrée et peut écraser `psi0` pendant les itérations. L'exemple lui fournit donc une copie. Le chemin compilé applique le même critère de convergence, mais ne construit pas l'objet résultat ni sa validation finale.

In [ ]:
psi_fast, phi_fast, iterations_fast, error_fast = sn_problem.solve_flux_only(
    psi0=psi_guess,
    fixed_source=q_angular,
    source_operator=mesh.Sigma_p,
    phi_tol=PHI_TOL,
)
psi_trusted, phi_trusted, iterations_trusted, error_trusted = sn_problem.solve_flux_only(
    psi0=psi_guess.copy(),
    fixed_source=q_angular,
    source_operator=mesh.Sigma_p,
    phi_tol=PHI_TOL,
    assume_valid=True,
)

np.testing.assert_allclose(psi_fast, sn_stationary.psi, rtol=1e-10, atol=1e-12)
np.testing.assert_allclose(phi_fast, phi_sn, rtol=1e-10, atol=1e-12)
np.testing.assert_allclose(phi_trusted, phi_fast, rtol=1e-10, atol=1e-12)
assert iterations_fast == iterations_trusted == sn_stationary.iterations
assert error_fast < PHI_TOL and error_trusted < PHI_TOL

print(f"Écart relatif chemin complet / compilé : {relative_gap(phi_fast, phi_sn):.3e}")
print(f"Écart relatif validé / assume_valid : {relative_gap(phi_trusted, phi_fast):.3e}")
print(f"Itérations : {iterations_fast} ; erreur finale : {error_fast:.3e}")

## 7. Transport transitoire — `EulerTransportProblem`

**Problème.** On reprend la source constante et le flux initial nul :

$$
\frac1v\partial_t\psi+\mu\partial_x\psi+\Sigma_t\psi
=\frac{\Sigma_p}{2}\phi+\frac q2,\qquad \psi(x,\mu,0)=0.
$$

**Euler implicite et itération interne.** En posant $\alpha=1/(v\Delta t)$, chaque nouveau pas vérifie

$$
\mu\partial_x\psi^{n+1}+(\Sigma_t+\alpha)\psi^{n+1}
=\frac{\Sigma_p}{2}\phi^{n+1}+\frac q2+\alpha\psi^n.
$$

Le balayage de l'itération interne $\ell$ est donc

$$
\psi_{m,i}^{n+1,\ell+1}
=\frac{|\mu_m|\psi_{m,\mathrm{amont}(i)}^{n+1,\ell+1}
+h\left(\frac{\Sigma_{p,i}}2\phi_i^{n+1,\ell}+\frac{q_i}2
+\alpha\psi_{m,i}^{n}\right)}
{|\mu_m|+h(\Sigma_{t,i}+\alpha)}.
$$

L'ancien état $\psi^n$ reste **fixe pendant toutes les itérations internes**. Le critère sur la variation du flux scalaire est identique au stationnaire et doit être satisfait à chaque pas.

**Convention de source dans le code actuel.** Le défaut de `EulerTransportProblem` utilise `mesh.source` directement comme source angulaire, alors que le solveur stationnaire utilise sa moitié. Pour comparer exactement la même source physique, on passe explicitement `fixed_source=0.5 * mesh.source`.

**Stockage.** `PHI` contient tous les pas, état initial compris. `PSI` et `Current` sont conservés tous les dix pas par défaut, ainsi qu'au dernier pas. Les instants correspondants sont `stored_steps * timestep`.

In [ ]:
sn_time_problem = EulerTransportProblem(
    mesh=mesh,
    quadrature=quadrature,
    velocity=reactor.speed,
    endtime=ENDTIME,
    timestep=TIMESTEP,
)
sn_time = sn_time_problem.solve(
    psi0=np.zeros((quadrature.order, mesh.n_cells)),
    fixed_source=q_angular,
    source_operator=mesh.Sigma_p,
    phi_tol=PHI_TOL,
)
stored_times = sn_time.stored_steps * TIMESTEP
sn_time_gap = np.max(np.abs(sn_time.PHI - phi_sn[None, :]), axis=1) / np.max(np.abs(phi_sn))

assert sn_time.PHI.shape == (N_STEPS + 1, mesh.n_cells)
assert sn_time.PSI.shape == (len(stored_times), quadrature.order, mesh.n_cells)
assert sn_time.is_finite and sn_time.is_positive
assert np.max(sn_time.errors) < PHI_TOL
assert sn_time_gap[-1] < sn_time_gap[0]
np.testing.assert_allclose(
    np.einsum("m,tmi->ti", quadrature.weights, sn_time.PSI),
    sn_time.PHI[sn_time.stored_steps],
    rtol=1e-12,
    atol=1e-12,
)

print("PHI :", sn_time.PHI.shape, "| PSI :", sn_time.PSI.shape)
print("Instants de stockage angulaire :", stored_times)
print(f"Plus grande erreur d'itération interne : {np.max(sn_time.errors):.3e}")
print(f"Écart relatif final au stationnaire SN : {sn_time_gap[-1]:.3e}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), layout="constrained")
shade_reactor(axes[0])
for step in snapshot_steps:
    axes[0].plot(x, sn_time.PHI[step], label=f"t = {times[step]:g}")
axes[0].plot(x, phi_sn, "k--", lw=1.3, label="Stationnaire SN")
axes[0].set_ylabel("Flux scalaire")
axes[0].set_title("Démarrage du transport")
axes[0].legend(fontsize=8)

axes[1].semilogy(times, diffusion_time_gap, "--", label="Diffusion")
axes[1].semilogy(times, sn_time_gap, label="Transport SN")
axes[1].set_xlabel("Temps")
axes[1].set_ylabel("Écart relatif au stationnaire")
axes[1].set_title("Relaxation vers les stationnaires")
axes[1].legend()

axes[2].plot(times[1:], sn_time.iterations, color="#0d9488")
axes[2].set_xlabel("Temps")
axes[2].set_ylabel("Itérations internes")
axes[2].set_title("Coût de convergence par pas")
plt.show()

## 8. Construire la criticité en transport avec les outils SN

Cette section ajoute une **boucle pédagogique dans le notebook**, en combinant `solve_flux_only` et `criticality_residual`.

**Problème propre, sans source externe :**

$$
\mu\,\partial_x\psi+\Sigma_t\psi
=\frac{\Sigma_s}{2}\phi+\frac{\nu\Sigma_f}{2k}\phi.
$$

**Deux niveaux d'itération.** À l'itération externe $n$, on fige la source de fission et on résout par itérations internes

$$
\mu\,\partial_x\widetilde\psi+\Sigma_t\widetilde\psi
=\frac{\Sigma_s}{2}\widetilde\phi
+\underbrace{\frac{\nu\Sigma_f}{2k^{(n)}}\phi^{(n)}}_{\text{source de fission fixée}}.
$$

Ainsi `source_operator=mesh.Sigma_s` conserve uniquement le couplage par diffusion angulaire dans la résolution interne. Passer $\Sigma_p$ compterait la production de fission une seconde fois.

Pour $P(\phi^{(n)})=1$, la mise à jour externe est

$$
k^{(n+1)}=k^{(n)}P(\widetilde\phi),\qquad
\phi^{(n+1)}=\frac{\widetilde\phi}{P(\widetilde\phi)},\qquad
\psi^{(n+1)}=\frac{\widetilde\psi}{P(\widetilde\phi)}.
$$

Le mode de diffusion initialise la forme spatiale. La boucle ci-dessous utilise une tolérance interne fixe pour rester lisible.

**Résidu propre.** Si $\mathcal L_h$ est l'opérateur discret transport moins diffusion angulaire et $\mathcal F_h$ la production de fission, `criticality_residual` calcule

$$
r_k=\frac{\|\mathcal L_h\psi-\mathcal F_h\psi/k\|_2}
{\max(\|\mathcal L_h\psi\|_2+\|\mathcal F_h\psi/k\|_2,\varepsilon_{\rm tiny})}.
$$

On arrête lorsque la variation relative de $k$ et ce résidu sont suffisamment petits. La source externe $q$ du cas stationnaire est remplacée par la source propre de fission.

In [ ]:
phi_eigen_sn = criticality.phi.copy()
psi_eigen_sn = np.broadcast_to(
    0.5 * phi_eigen_sn, (quadrature.order, mesh.n_cells)
).copy()
k_sn = criticality.k_eff
k_history, residual_history, inner_history = [], [], []

for outer_iteration in range(1, 501):
    fixed_fission_source = 0.5 * fission * phi_eigen_sn / k_sn
    psi_raw, phi_raw, inner_iterations, inner_error = sn_problem.solve_flux_only(
        psi0=psi_eigen_sn,
        fixed_source=fixed_fission_source,
        source_operator=mesh.Sigma_s,
        phi_tol=1e-11,
    )
    production = mesh.h * np.dot(fission, phi_raw)
    if not np.isfinite(production) or production <= 0.0:
        raise RuntimeError("Production de fission invalide.")

    k_new = k_sn * production
    phi_eigen_sn = phi_raw / production
    psi_eigen_sn = psi_raw / production
    relative_k_change = abs(k_new - k_sn) / abs(k_new)
    k_sn = k_new
    sn_eigen_residual = sn_problem.criticality_residual(
        psi_eigen_sn, phi_eigen_sn, k_sn
    )
    k_history.append(k_sn)
    residual_history.append(sn_eigen_residual)
    inner_history.append(inner_iterations)

    if relative_k_change < 1e-9 and sn_eigen_residual < 1e-8:
        break
else:
    raise RuntimeError("La criticité SN n'a pas convergé en 500 itérations.")

assert 0 < k_sn < 1
assert np.all(psi_eigen_sn >= 0)
assert sn_eigen_residual < 1e-8
np.testing.assert_allclose(mesh.h * np.dot(fission, phi_eigen_sn), 1.0, rtol=1e-12)
np.testing.assert_allclose(quadrature.weights @ psi_eigen_sn, phi_eigen_sn)

print(f"k_eff diffusion = {k_diffusion:.8f}")
print(f"k_eff transport = {k_sn:.8f}")
print(f"Itérations externes : {outer_iteration} ; internes cumulées : {sum(inner_history)}")
print(f"Résidu propre SN : {sn_eigen_residual:.3e}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), layout="constrained")
shade_reactor(axes[0])
axes[0].plot(x, criticality.phi / criticality.phi.max(), "--", label="Diffusion")
axes[0].plot(x, phi_eigen_sn / phi_eigen_sn.max(), label="Transport SN")
axes[0].set_ylabel("Mode normalisé par son maximum")
axes[0].set_title("Modes fondamentaux")
axes[0].legend()

outer_steps = np.arange(1, len(k_history) + 1)
axes[1].plot(outer_steps, k_history, "o-", markersize=3, label="Transport SN")
axes[1].axhline(k_diffusion, color="#d97706", ls="--", label="Diffusion")
axes[1].set_xlabel("Itération externe")
axes[1].set_ylabel(r"$k_{\mathrm{eff}}$")
axes[1].set_title("Facteur de multiplication")
axes[1].legend()

axes[2].semilogy(outer_steps, residual_history, "o-", markersize=3)
axes[2].axhline(1e-8, color="#d97706", ls="--", label="Tolérance")
axes[2].set_xlabel("Itération externe")
axes[2].set_ylabel("Résidu propre normalisé")
axes[2].set_title("Vérification de l'équation propre")
axes[2].legend()
plt.show()

## Bilan : lire les résultats et reprendre le projet

La table suivante rassemble les quantités calculées. Les résidus absolus et relatifs portent sur des équations différentes : ils ne constituent pas un classement des solveurs. L'écart diffusion/SN traduit à la fois l'approximation physique, les conditions aux limites et la discrétisation.

In [ ]:
display(Markdown(f"""
| Calcul | Indicateur obtenu |
| --- | --- |
| Source imposée, sans production | Résidu matriciel absolu : {source_residual:.2e} |
| Diffusion stationnaire avec fission | Résidu de bilan local : {diffusion_stationary.residual_norm_inf:.2e} |
| Criticité diffusion | k_eff = {k_diffusion:.8f} ; résidu relatif : {eigen_residual:.2e} |
| Diffusion transitoire | Écart final au stationnaire : {diffusion_time_gap[-1]:.2e} |
| Transport stationnaire | {sn_stationary.iterations} itérations ; résidu discret absolu : {sn_residual_max:.2e} |
| Transport compilé | Écart relatif au chemin complet : {relative_gap(phi_fast, phi_sn):.2e} |
| Transport transitoire | Écart final au stationnaire : {sn_time_gap[-1]:.2e} |
| Criticité SN, boucle du notebook | k_eff = {k_sn:.8f} ; résidu relatif : {sn_eigen_residual:.2e} |
| Comparaison des flux stationnaires | Écart relatif diffusion/SN : {diffusion_sn_gap:.2%} |
"""))

### Trois expériences pour se réapproprier le code

1. **Maillage et quadrature.** Passer de 120 à 240 cellules, puis de 8 à 16 directions. Relancer toutes les cellules et observer séparément l'évolution de $k_{\mathrm{eff}}$, des profils et du nombre d'itérations. Garder les interfaces cœur–réflecteur sur des faces.
2. **Temps.** Réduire `TIMESTEP` de 2 à 1 à durée finale identique ; comparer les profils à un même instant. Réduire plutôt `ENDTIME` montre un état encore éloigné de l'équilibre.
3. **Physique.** Modifier légèrement la capture dans le cœur ou l'épaisseur du réflecteur, puis comparer les facteurs de multiplication. Les démos à source constante supposent un régime sous-critique ; le contrôle initial $\Sigma_a>\nu\Sigma_f$ est une condition suffisante choisie pour ce parcours, pas une condition nécessaire de sous-criticité.

**Points à retenir pour utiliser les API :** source scalaire $q$, source angulaire $q/2$ et second membre $hq$ ; flux propre normalisé versus flux avec source ; courants aux faces versus aux centres ; temps de stockage angulaire distincts des temps de stockage scalaire.

Pour relire l'implémentation : [opérateurs de diffusion](../transport/diffusion/operators.py), [diffusion stationnaire et criticité](../transport/diffusion/stationary.py), [diffusion transitoire](../transport/diffusion/transient.py), [transport stationnaire](../transport/sn/stationary.py), [transport transitoire](../transport/sn/transient.py), [noyaux compilés](../transport/sn/_kernels.py).